[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aims-foundations/torch_measure/blob/main/tutorials/amortized_irt.ipynb)

# Amortized IRT: Predicting Item Difficulty from Question Text

This tutorial demonstrates how to train an **Amortized IRT** model that predicts item parameters (difficulty, discrimination) directly from question embeddings. This enables **zero-shot calibration** of new items without collecting additional response data.

The approach follows [Truong et al. (2025)](https://arxiv.org/abs/2503.13335), which uses a neural network to map question embeddings to IRT parameters via an EM-style training procedure. The key insight: question difficulty is **predictable from the question text itself**, meaning we can calibrate new questions without expensive human or LLM evaluation.

**What you'll learn:**
1. Loading HELM benchmark data using `torch_measure.datasets`
2. Generating item embeddings from question text using a sentence encoder
3. Training an `AmortizedIRT` model that maps embeddings to item parameters
4. Evaluating in-sample fit and zero-shot generalization to held-out items
5. Inspecting the learned difficulty and discrimination parameters

## 1. Setup

In [ ]:
try:
    import google.colab
    !git clone https://github.com/aims-foundations/torch_measure.git
    !pip install -e "torch_measure[data]" sentence-transformers
except ImportError:
    pass  # Already installed locally

import torch
import matplotlib.pyplot as plt
import numpy as np

from torch_measure.datasets import load, list_datasets, info
from torch_measure.data import ResponseMatrix, random_mask, col_mask
from torch_measure.models import AmortizedIRT, Rasch, TwoPL
from torch_measure.metrics import expected_calibration_error, brier_score

plt.rcParams["figure.dpi"] = 120
torch.manual_seed(42)
print("Setup complete.")

## 2. Load a HELM Benchmark Dataset

`torch_measure` ships response matrices from the HELM evaluation suite — 183 LLMs evaluated across 22 NLP benchmarks. Each dataset contains:
- **Response matrix**: Binary correctness (183 LLMs × N items)
- **Item contents**: The raw question text for each item
- **Subject metadata**: Model name, organization, parameter count, etc.

We'll use MMLU (Massive Multitask Language Understanding) as our running example.

In [ ]:
# See what's available
print("Available HELM datasets:")
for name in list_datasets(family="helm"):
    ds_info = info(name)
    print(f"  {name:30s}  {ds_info.n_items:>6,} items  —  {ds_info.description}")

In [ ]:
# Load MMLU
rm = load("helm/mmlu")
print(rm)
print(f"Density: {rm.density:.1%}")
print(f"Overall accuracy: {rm.data[rm.observed_mask].mean():.3f}")
print(f"\nFirst 3 subjects: {rm.subject_ids[:3]}")
print(f"First 3 item IDs:  {rm.item_ids[:3]}")
print(f"\nSample question (item 0):")
print(rm.item_contents[0][:300], "..." if len(rm.item_contents[0]) > 300 else "")

In [ ]:
# Inspect subject metadata
print("Subject metadata fields:", list(rm.subject_metadata[0].keys()))
print("\nSample subjects:")
for i in range(5):
    meta = rm.subject_metadata[i]
    print(f"  {rm.subject_ids[i]:40s}  org={meta['org']}, params={meta.get('param_count', '?')}")

## 3. Generate Item Embeddings

The amortized model needs a vector representation of each question. [Truong et al. (2025)](https://arxiv.org/abs/2503.13335) use Llama-3-8B embeddings (dim 4096). For this tutorial, we use a lighter `sentence-transformers` model that runs on CPU.

Any encoder works — the `AmortizedIRT` model just needs a `(n_items, embedding_dim)` tensor.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a lightweight sentence encoder
encoder = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim, ~80MB

# Encode all question texts
print(f"Encoding {rm.n_items} items...")
embeddings = encoder.encode(
    rm.item_contents,
    show_progress_bar=True,
    convert_to_tensor=True,  # returns torch.Tensor directly
).clone()

print(f"Embeddings shape: {embeddings.shape}")  # (n_items, 384)


## 4. Train/Test Split: Held-Out Items

The key test for amortized IRT is **zero-shot generalization to new items**. We use a column-based (item-based) mask: 80% of items are used for training, 20% are held out entirely. The model must predict response probabilities for the held-out items using only their embeddings.

This is the cold-start scenario from [Truong et al. (2025)](https://arxiv.org/abs/2503.13335) — new questions that no LLM has been evaluated on yet.

In [ ]:
torch.manual_seed(42)

# Column-based split: hold out 20% of items completely
n_items = rm.n_items
perm = torch.randperm(n_items)
n_train_items = int(0.8 * n_items)

train_item_idx = perm[:n_train_items].sort().values
test_item_idx = perm[n_train_items:].sort().values

print(f"Train items: {len(train_item_idx)}")
print(f"Test items:  {len(test_item_idx)} (held-out, zero-shot)")

# Create the train/test response matrices and embeddings
train_responses = rm.data[:, train_item_idx]
test_responses = rm.data[:, test_item_idx]
train_embeddings = embeddings[train_item_idx]
test_embeddings = embeddings[test_item_idx]

## 5. Train the Amortized IRT Model

The `AmortizedIRT` model learns:
- **Subject abilities** $\theta_i$ — a free parameter per LLM (learned directly)
- **Item parameter network** $f_\phi$ — an MLP that maps embeddings to difficulty $b_j$ and discrimination $a_j$

$$P(\text{correct}_{ij}) = \sigma\big(a_j (\theta_i - b_j)\big), \quad (b_j, a_j) = f_\phi(\mathbf{e}_j)$$

All parameters are trained jointly via maximum likelihood.

In [ ]:
n_subjects = rm.n_subjects
embedding_dim = embeddings.shape[1]

amort = AmortizedIRT(
    n_subjects=n_subjects,
    n_items=len(train_item_idx),
    embedding_dim=embedding_dim,
    hidden_dim=256,
    n_layers=3,
    pl=2,         # 2PL: learns difficulty + discrimination
    dropout=0.1,
)

history = amort.fit(
    train_responses,
    train_embeddings,
    max_epochs=500,
    lr=1e-3,
    weight_decay=1e-4,
    verbose=True,
)

print(f"\nFinal training loss: {history['losses'][-1]:.4f}")

In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(history["losses"])
ax.set_xlabel("Epoch")
ax.set_ylabel("Negative Log-Likelihood")
ax.set_title("Amortized IRT Training Loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Baseline: Standard 2PL (Non-Amortized)

For comparison, we fit a standard 2PL model on the same training items. This model learns independent difficulty/discrimination parameters per item — it **cannot** generalize to new items.

In [ ]:
baseline = TwoPL(n_subjects=n_subjects, n_items=len(train_item_idx))
history_baseline = baseline.fit(train_responses, max_epochs=500, verbose=False)
print(f"Baseline 2PL final loss: {history_baseline['losses'][-1]:.4f}")
print(f"Amortized final loss:    {history['losses'][-1]:.4f}")

## 7. In-Sample Evaluation

Compare how well the amortized and baseline models fit the training data.

In [ ]:
# In-sample predictions
with torch.no_grad():
    probs_amort_train = amort.predict()
    probs_baseline_train = baseline.predict()

train_mask = ~torch.isnan(train_responses)

ece_amort = expected_calibration_error(probs_amort_train, train_responses, mask=train_mask)
ece_base = expected_calibration_error(probs_baseline_train, train_responses, mask=train_mask)
bs_amort = brier_score(probs_amort_train, train_responses, mask=train_mask)
bs_base = brier_score(probs_baseline_train, train_responses, mask=train_mask)

print(f"{'Metric':<30s} {'Amortized':>12s} {'Baseline 2PL':>12s}")
print("-" * 56)
print(f"{'Brier Score (train)':30s} {bs_amort:12.4f} {bs_base:12.4f}")
print(f"{'ECE (train)':30s} {ece_amort:12.4f} {ece_base:12.4f}")

## 8. Zero-Shot Evaluation on Held-Out Items

This is where amortized IRT shines. We create a new model instance with the held-out items, transfer the learned MLP weights, and predict — **no retraining needed**.

The standard 2PL baseline cannot do this at all, since its item parameters are independent scalars with no way to generalize.

In [ ]:
# Create a new AmortizedIRT for the test items,
# reusing the learned item network and subject abilities
amort_test = AmortizedIRT(
    n_subjects=n_subjects,
    n_items=len(test_item_idx),
    embedding_dim=embedding_dim,
    hidden_dim=256,
    n_layers=3,
    pl=2,
    dropout=0.1,
)

# Transfer the learned item network (MLP) and subject abilities
amort_test.item_net.load_state_dict(amort.item_net.state_dict())
with torch.no_grad():
    amort_test.ability.copy_(amort.ability)

# Set the held-out item embeddings
amort_test.set_embeddings(test_embeddings)

# Predict response probabilities for unseen items
with torch.no_grad():
    probs_amort_test = amort_test.predict()

test_mask = ~torch.isnan(test_responses)
ece_test = expected_calibration_error(probs_amort_test, test_responses, mask=test_mask)
bs_test = brier_score(probs_amort_test, test_responses, mask=test_mask)

print(f"Zero-shot evaluation on {len(test_item_idx)} held-out items:")
print(f"  Brier Score: {bs_test:.4f}")
print(f"  ECE:         {ece_test:.4f}")

In [ ]:
# Compare: naive baselines for the test items
# 1. Global mean (predict average accuracy for everything)
global_mean = train_responses[~torch.isnan(train_responses)].mean()
bs_naive = brier_score(
    global_mean.expand_as(test_responses), test_responses, mask=test_mask
)

# 2. Per-subject mean (predict each LLM's average training accuracy)
subject_means = train_responses.clone()
subject_means[torch.isnan(subject_means)] = 0.0
train_obs = (~torch.isnan(train_responses)).float()
subject_avg = (subject_means.sum(dim=1) / train_obs.sum(dim=1).clamp(min=1)).unsqueeze(1)
bs_subject = brier_score(
    subject_avg.expand_as(test_responses), test_responses, mask=test_mask
)

print(f"{'Method':<30s} {'Brier Score':>12s}")
print("-" * 44)
print(f"{'Global mean':30s} {bs_naive:12.4f}")
print(f"{'Per-subject mean':30s} {bs_subject:12.4f}")
print(f"{'Amortized IRT (zero-shot)':30s} {bs_test:12.4f}")

## 9. Inspect Learned Parameters

Let's look at what the model learned — both the subject abilities (LLM rankings) and the predicted item parameters.

In [ ]:
# Subject abilities: rank the LLMs
abilities = amort.ability.detach()
ranking = abilities.argsort(descending=True)

print("Top 10 LLMs by estimated ability:")
for rank, idx in enumerate(ranking[:10]):
    name = rm.subject_ids[idx]
    meta = rm.subject_metadata[idx]
    print(f"  {rank+1:2d}. {name:40s}  theta={abilities[idx]:.3f}  ({meta.get('param_count', '?')} params)")

print("\nBottom 5 LLMs:")
for rank, idx in enumerate(ranking[-5:]):
    name = rm.subject_ids[idx]
    print(f"  {rm.n_subjects - 4 + rank:3d}. {name:40s}  theta={abilities[idx]:.3f}")

In [ ]:
# Ability distribution
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(abilities.numpy(), bins=30, edgecolor="black", alpha=0.7)
ax.set_xlabel("Estimated Ability (θ)")
ax.set_ylabel("Count")
ax.set_title("Distribution of LLM Abilities")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Item difficulty vs. empirical item means (facility)
difficulty_train = amort.difficulty
empirical_facility = rm.item_means[train_item_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Difficulty vs empirical facility
axes[0].scatter(empirical_facility.numpy(), difficulty_train.numpy(), alpha=0.15, s=8)
axes[0].set_xlabel("Empirical Facility (% correct)")
axes[0].set_ylabel("Predicted Difficulty (b)")
axes[0].set_title("Predicted Difficulty vs. Empirical Facility")
axes[0].grid(True, alpha=0.3)

# Discrimination distribution
disc_train = amort.discrimination
axes[1].hist(disc_train.numpy(), bins=40, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Predicted Discrimination (a)")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Predicted Discriminations")
axes[1].axvline(x=1.0, color="red", linestyle="--", alpha=0.5, label="a=1 (Rasch)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Zero-Shot Difficulty Prediction Quality

How well does the amortized model predict difficulty for **unseen** items? We compare the predicted difficulty to the empirical item facility on the held-out set.

In [ ]:
# Predicted difficulty for held-out items (zero-shot)
difficulty_test = amort_test.difficulty
empirical_facility_test = rm.item_means[test_item_idx]

# Correlation: predicted difficulty vs empirical facility
# Higher difficulty should correspond to lower facility
valid = ~torch.isnan(empirical_facility_test)
r = torch.corrcoef(torch.stack([difficulty_test[valid], empirical_facility_test[valid]]))[0, 1]

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(
    empirical_facility_test[valid].numpy(),
    difficulty_test[valid].numpy(),
    alpha=0.3, s=10,
)
ax.set_xlabel("Empirical Facility (% correct, held-out items)")
ax.set_ylabel("Zero-Shot Predicted Difficulty (b)")
ax.set_title(f"Zero-Shot Difficulty Prediction (r = {r:.3f})")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Pearson correlation (difficulty vs. facility): {r:.3f}")
print("(Negative correlation expected: harder items have lower facility)")

## 11. Scaling Up: Multiple Benchmarks

Following [Truong et al. (2025)](https://arxiv.org/abs/2503.13335), we can train a **global** amortized model across multiple benchmarks. This is straightforward — just load and concatenate.

In [ ]:
# Example: load multiple benchmarks and combine
benchmarks = ["helm/mmlu", "helm/gsm8k", "helm/truthfulqa"]
all_responses = []
all_contents = []

for name in benchmarks:
    ds = load(name)
    all_responses.append(ds.data)
    all_contents.extend(ds.item_contents)
    print(f"  {name:25s}  {ds.n_items:>6,} items")

# Concatenate along the item dimension (subjects are shared)
combined_responses = torch.cat(all_responses, dim=1)
print(f"\nCombined: {combined_responses.shape[0]} subjects x {combined_responses.shape[1]} items")
print(f"Density: {(~torch.isnan(combined_responses)).float().mean():.1%}")

In [ ]:
# Encode all items
print(f"Encoding {len(all_contents)} items across {len(benchmarks)} benchmarks...")
combined_embeddings = encoder.encode(
    all_contents,
    show_progress_bar=True,
    convert_to_tensor=True,
).clone()
print(f"Embeddings shape: {combined_embeddings.shape}")


In [ ]:
# Train a global amortized model
global_model = AmortizedIRT(
    n_subjects=combined_responses.shape[0],
    n_items=combined_responses.shape[1],
    embedding_dim=combined_embeddings.shape[1],
    hidden_dim=256,
    n_layers=3,
    pl=2,
    dropout=0.1,
)

history_global = global_model.fit(
    combined_responses,
    combined_embeddings,
    max_epochs=300,
    lr=1e-3,
    verbose=True,
)

print(f"\nGlobal model final loss: {history_global['losses'][-1]:.4f}")

## 12. Using Stronger Embeddings

[Truong et al. (2025)](https://arxiv.org/abs/2503.13335) use Llama-3-8B embeddings (dim 4096). You can swap in any encoder — stronger embeddings generally yield better difficulty prediction. Here's how you'd use a different encoder:

In [ ]:
# Example with a different encoder (not executed — just showing the pattern)
# Any encoder that maps text -> fixed-size vectors works.
#
# Option 1: Larger sentence-transformers model
# encoder = SentenceTransformer("all-mpnet-base-v2")  # 768-dim
#
# Option 2: Instruction-tuned embeddings
# encoder = SentenceTransformer("BAAI/bge-large-en-v1.5")  # 1024-dim
#
# Option 3: LLM embeddings (as in the paper)
# from transformers import AutoModel, AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B")
# llm = AutoModel.from_pretrained("meta-llama/Meta-Llama-3-8B")
# # Use last hidden state mean pooling as embedding
#
# The AmortizedIRT model adapts to any embedding_dim automatically.

print("Swap in any encoder — just change embedding_dim in AmortizedIRT accordingly.")

## Summary

In this tutorial we:

1. **Loaded** a HELM benchmark dataset using `torch_measure.datasets.load()`
2. **Embedded** question text using `sentence-transformers`
3. **Trained** an `AmortizedIRT` model mapping embeddings → (difficulty, discrimination)
4. **Evaluated** zero-shot generalization to held-out items
5. **Compared** against a standard 2PL baseline and naive predictors
6. **Scaled** to multiple benchmarks with a global model

The amortized approach is especially valuable when:
- You have a large bank of calibrated items and want to add new ones cheaply
- You want to reduce evaluation cost by predicting which items are informative
- You need difficulty estimates for items before collecting any response data

### References

- Truong et al. (2025). *Model-based Evaluation and Diagnosis of Language Model.* [arXiv:2503.13335](https://arxiv.org/abs/2503.13335)
- Liang et al. (2023). *Holistic Evaluation of Language Models.* TMLR.
- Lord, F. M. (1980). *Applications of Item Response Theory to Practical Testing Problems.* Erlbaum.